In [1]:
from pathlib import Path

project_root = Path.cwd().parent
data_dir = project_root / "data" / "raw"

In [2]:
import sys
sys.path.append('..')
from src.bookstore_health_analysis.data_loader import load_all_data

dfs = load_all_data(data_dir)

In [3]:
for f in dfs:
    print(f"Dataframe: {f}")

Dataframe: bookstore_checking_balanced_dataset
Dataframe: bookstore_credit_balance_dataset
Dataframe: bookstore_customers
Dataframe: bookstore_customers_expanded
Dataframe: bookstore_customers_expanded_raw
Dataframe: bookstore_employees
Dataframe: bookstore_loans
Dataframe: bookstore_payroll
Dataframe: bookstore_sales


In [4]:
dfs['bookstore_sales'].head()

,date,year,month,store_id,channel,transactions,revenue,gst_collected,net_revenue,dataset
0,2019-01-01,2019,1,YYC-DT,IN_STORE,64,1985.46,99.27,1886.19,bookstore_sales
1,2019-01-01,2019,1,YYC-DT,ONLINE,8,207.72,10.39,197.33,bookstore_sales
2,2019-01-01,2019,1,YYC-NW,IN_STORE,49,1466.95,73.35,1393.60,bookstore_sales
3,2019-01-01,2019,1,YYC-NW,ONLINE,4,148.89,7.44,141.45,bookstore_sales
4,2019-01-01,2019,1,YYC-SE,IN_STORE,33,1272.70,63.64,1209.06,bookstore_sales


In [10]:
sales_monthly = dfs['bookstore_sales'].groupby(
    [dfs['bookstore_sales']['date'].dt.to_period('Q')]).agg({'revenue': 'sum'}).reset_index()

sales_monthly

,date,revenue
0,2019Q1,516338.28
1,2019Q2,615524.98
2,2019Q3,616114.33
3,2019Q4,742654.91
4,2020Q1,365486.37
5,2020Q2,421398.42
6,2020Q3,431929.89
7,2020Q4,512393.58
8,2021Q1,441220.96
9,2021Q2,528340.59


In [11]:
checking_monthly = dfs['bookstore_checking_balanced_dataset'].groupby(
    [dfs['bookstore_checking_balanced_dataset']['date'].dt.to_period('Q')]).agg({'deposit': 'sum'}).reset_index()

checking_monthly

,date,deposit
0,2019Q1,126423.93
1,2019Q2,128143.23
2,2019Q3,142558.15
3,2019Q4,135841.49
4,2020Q1,93982.64
5,2020Q2,95084.72
6,2020Q3,96876.67
7,2020Q4,112293.19
8,2021Q1,111940.96
9,2021Q2,118968.07


In [7]:
dfs['bookstore_loans'].columns

Index(['quarter_end', 'loan_name', 'loan_id', 'principal_outstanding',
       'interest_accrued', 'repayment', 'dataset'],
      dtype='str')

In [8]:
loan_quarterly = dfs['bookstore_loans'].groupby(
    dfs['bookstore_loans']['quarter_end'].dt.to_period('Q')
).agg({'repayment': 'sum', 'principal_outstanding': 'last'}).reset_index()

loan_quarterly

,quarter_end,repayment,principal_outstanding
0,2019Q1,17187.85,10715.37
1,2019Q2,17245.21,2326.04
2,2019Q3,19206.88,0.00
3,2019Q4,21753.93,0.00
4,2020Q1,13342.21,15393.43
5,2020Q2,16802.95,16722.70
6,2020Q3,23546.66,5512.96
7,2020Q4,8336.41,14702.55
8,2021Q1,9383.03,30099.62
9,2021Q2,16257.44,24999.82


In [12]:
combined_ledger = sales_monthly.merge(checking_monthly, on='date', how='inner').merge(loan_quarterly, left_on='date', right_on='quarter_end', how='inner')
combined_ledger.head()

,date,revenue,deposit,quarter_end,repayment,principal_outstanding
0,2019Q1,516338.28,126423.93,2019Q1,17187.85,10715.37
1,2019Q2,615524.98,128143.23,2019Q2,17245.21,2326.04
2,2019Q3,616114.33,142558.15,2019Q3,19206.88,0.00
3,2019Q4,742654.91,135841.49,2019Q4,21753.93,0.00
4,2020Q1,365486.37,93982.64,2020Q1,13342.21,15393.43


In [13]:
combined_ledger = combined_ledger.drop(columns=['quarter_end'])